# Sprint Block N — Multilingual input: Urdu / Roman-Urdu → the same pipeline
Front-end only: Urdu or Roman-Urdu text → NLLB-200 translation to English → the unchanged extraction + ranking pipeline. Produces the translation-loss experiment on the Pakistan evaluation set (n≥3 subset) and a qualitative run on the 243 held Wikivoyage Urdu sentences. Writes `results_n/`.

## N0 — Translation sheet for the author
`eval/pk_translation_sheet.xlsx`: the n≥3 Pakistan sentences with empty `urdu` and `roman_urdu` columns. Fill `urdu` for every row (natural paraphrase, keep proper names, do not name the place type if the English does not); fill `roman_urdu` for as many rows as practical (≥ 60), written the way you would actually text it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/geoproject'

In [ ]:
# N0.1 — write the translation sheet (n>=3 subset of the adjudicated Pakistan set)
import json, os, pandas as pd
from collections import Counter
os.makedirs(f'{ROOT}/results_n', exist_ok=True)
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); d = Counter(r['gold_tag'] for r in pk)
rows = [r for r in pk if d[r['gold_tag']] >= 3]
sheet = pd.DataFrame({'row': [r['row'] for r in rows], 'sentence': [r['sentence'] for r in rows], 'urdu': '', 'roman_urdu': ''})
with pd.ExcelWriter(f'{ROOT}/eval/pk_translation_sheet.xlsx') as xw:
    sheet.to_excel(xw, sheet_name='translate', index=False)
    pd.DataFrame({'instruction': ['urdu: fill every row, Urdu script, natural paraphrase (as you would describe the place to a friend)', 'keep proper names (Clifton, Delhi Gate, F-7) as they are',
                                  'do not name the place type if the English sentence does not (no "restaurant"/"masjid" if the English avoids it)', 'roman_urdu: at least 60 rows, written the way you would text it; English words mixed in are fine',
                                  'save in place with the same file name']}).to_excel(xw, sheet_name='how_to', index=False)
sheet.to_csv(f'{ROOT}/eval/pk_translation_sheet.csv', index=False)
print(f'wrote eval/pk_translation_sheet.xlsx / .csv — {len(sheet)} rows ({len([t for t, c in d.items() if c >= 3])} tags); gold tags are NOT in the sheet on purpose')
print(sheet.head(3).to_string(index=False))

## N1 — Translate (GPU: T4)
Urdu script → English with **NLLB-200 (distilled 600M)**, the standard open MT baseline. Roman-Urdu has no dedicated MT model (Hugging Face hub search, Sept 2026: only sentiment/ASR models), so it is translated with an open instruction-tuned multilingual LLM run locally (**Qwen2.5-7B-Instruct, 4-bit**); the same LLM also translates the Urdu column so the two translators can be compared. Outputs `results_n/n1_translations.json`.

## N2 — Translation loss on the Pakistan evaluation set
Each translated English sentence goes through the unchanged pipeline (transformer extractor → nouns/adjectives → similarity + adjective→tag table, λ = 0.35). Compared with the English originals **through the same fully automatic path** (extractor nouns for both), so the difference is the translation loss and nothing else. Outputs `results_n/n2_translation_loss.json` (→ F4.8 / T4.15).

## N3 — The 243 held Wikivoyage Urdu / Roman-Urdu sentences
No gold labels exist for these; they are run through the front-end as a qualitative check (translation + top-3 tags + explanation) → `results_n/n3_held_urdu.json` (→ example table).

In [ ]:
# N1.1 — read the filled translation sheet + the 243 held sentences; load the two translators (GPU)
import json, os, time, re, subprocess, sys, torch, numpy as np, pandas as pd
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)
p_x, p_c = f'{ROOT}/eval/pk_translation_sheet.xlsx', f'{ROOT}/eval/pk_translation_sheet.csv'
if os.path.getmtime(p_x) > os.path.getmtime(p_c):
    book = pd.read_excel(p_x, sheet_name=None); sh = next(df for df in book.values() if {'row', 'urdu'} <= set(map(str, df.columns)))
else: sh = pd.read_csv(p_c)
for c in ('urdu', 'roman_urdu'): sh[c] = sh[c].fillna('').astype(str).str.strip()
sh = sh[sh.urdu != '']; print(f'sheet: {len(sh)} rows with Urdu, {int((sh.roman_urdu != "").sum())} with Roman-Urdu')
held = pd.read_parquet(f'{ROOT}/clean/held_urdu_bucket.parquet'); tcol = next(c for c in ('text', 'sentence') if c in held.columns)
is_arab = held[tcol].astype(str).str.contains(r'[؀-ۿ]'); print(f'held Urdu bucket: {len(held)} rows | Urdu script {int(is_arab.sum())} | Roman {int((~is_arab).sum())}')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
dev = 'cuda'; assert torch.cuda.is_available(), 'switch the runtime to a GPU (T4 is enough)'
nl_tok = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M', src_lang='urd_Arab'); nl = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M', dtype=torch.float16).to(dev).eval()
def nllb(texts, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        enc = nl_tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=256).to(dev)
        with torch.no_grad(): gen = nl.generate(**enc, forced_bos_token_id=nl_tok.convert_tokens_to_ids('eng_Latn'), max_new_tokens=160, num_beams=4)
        out += nl_tok.batch_decode(gen, skip_special_tokens=True)
    return out
qw_id = 'Qwen/Qwen2.5-7B-Instruct'
qw_tok = AutoTokenizer.from_pretrained(qw_id); qw = AutoModelForCausalLM.from_pretrained(qw_id, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'), device_map='auto').eval()
def llm_translate(texts, lang, bs=8):
    out = []; qw_tok.padding_side = 'left'
    for i in range(0, len(texts), bs):
        msgs = [[{'role': 'system', 'content': 'You are a translator. Translate the user text into natural English. Keep place names as they are. Output only the translation.'}, {'role': 'user', 'content': f'{lang} text: {t}'}] for t in texts[i:i+bs]]
        enc = qw_tok([qw_tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs], return_tensors='pt', padding=True).to(dev)
        with torch.no_grad(): gen = qw.generate(**enc, max_new_tokens=120, do_sample=False)
        out += [s.strip().split('\n')[0] for s in qw_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
    return out
print('translators ready:', nl.__class__.__name__, '+', qw_id)

In [ ]:
# N1.2 — translate the sheet (Urdu: NLLB + LLM; Roman-Urdu: LLM) and the 243 held sentences → results_n/n1_translations.json
t0 = time.time()
T = {'row': sh.row.astype(int).tolist(), 'urdu': sh.urdu.tolist(), 'roman_urdu': sh.roman_urdu.tolist()}
T['en_from_urdu_nllb'] = nllb(T['urdu']); T['en_from_urdu_llm'] = llm_translate(T['urdu'], 'Urdu')
ru_idx = [i for i, s in enumerate(T['roman_urdu']) if s]; ru_en = llm_translate([T['roman_urdu'][i] for i in ru_idx], 'Roman Urdu (Urdu written in Latin letters, possibly mixed with English)')
T['en_from_roman_llm'] = ['' for _ in T['row']]
for i, s in zip(ru_idx, ru_en): T['en_from_roman_llm'][i] = s
H = {'text': held[tcol].astype(str).tolist(), 'script': ['urdu' if a else 'roman' for a in is_arab]}
H['en_nllb'] = ['' for _ in H['text']]; ua = [i for i, s in enumerate(H['script']) if s == 'urdu']
for i, s in zip(ua, nllb([H['text'][i] for i in ua])): H['en_nllb'][i] = s
H['en_llm'] = llm_translate(H['text'], 'Urdu or Roman Urdu')
json.dump({'sheet': T, 'held': H, 'models': {'nllb': 'facebook/nllb-200-distilled-600M (urd_Arab→eng_Latn, 4 beams)', 'llm': qw_id + ' (4-bit, greedy)'}, 'seconds': time.time() - t0}, open(f'{ROOT}/results_n/n1_translations.json', 'w'), ensure_ascii=False, indent=1)
print(f'translated {len(T["row"])} Urdu + {len(ru_idx)} Roman-Urdu sheet rows and {len(H["text"])} held sentences in {time.time()-t0:.0f}s')
for i in range(3): print(f'\n[{T["row"][i]}] UR: {T["urdu"][i][:80]}\n   NLLB: {T["en_from_urdu_nllb"][i][:100]}\n   LLM : {T["en_from_urdu_llm"][i][:100]}' + (f'\n   RU→ : {T["en_from_roman_llm"][i][:100]}' if T['en_from_roman_llm'][i] else ''))

In [ ]:
# N2.0 — the ranker as an importable module (same code as the figures notebook; also used by the web app). Written once to ROOT/pipeline_ranker.py
RANKER_SRC = r'''
"""Ranking stage of the pipeline: context nouns -> embedding similarity to the 227 OSM tags, blended with the mined adjective->tag table.
Identical to the evaluation code; loads the small result files from ROOT (or an assets dir)."""
import json, pickle, re, numpy as np
from collections import Counter
ABSTRACT = {'place','spot','location','area','site','position','thing','kind','sort','lot','bit','way','day','time'}
KS, LAM = (1, 2, 3, 5, 10, 20), 0.35
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def variants(s):
    b = [s, s.lower(), s.title(), s.capitalize()]; o, seen = [], set()
    for v in b + [x.replace(' ', '_').replace('-', '_') for x in b] + [x.replace(' ', '').replace('-', '') for x in b]:
        if v and v not in seen: seen.add(v); o.append(v)
    return o
def ladder(VEC):
    def f(s):
        for v in variants(s):
            if v in VEC: return VEC[v]
        sub = [VEC[v] for w in parts(s) for v in variants(w) if v in VEC][:8]
        return np.mean(sub, 0) if sub else None
    return f
class Tables:
    """adjective->tag table (elastic net) + pattern->tag table (Bayesian elastic net posterior)"""
    def __init__(self, adj_npz, pat_npz, pat_meta_npz):
        e = np.load(adj_npz, allow_pickle=True); self.COEF, self.EN_CLS, self.EN_ADJ = e['coef'], list(e['classes']), list(e['adj_vocab']); self.AIX = {a: j for j, a in enumerate(self.EN_ADJ)}
        b = np.load(pat_npz, allow_pickle=True); m = np.load(pat_meta_npz, allow_pickle=True)
        self.PV, self.PCLS, self.NADJ = list(m['patvocab']), list(m['classes']), int(m['n_adj']); self.PIX = {p: j for j, p in enumerate(self.PV)}
        self.ADJ_SET, self.PAIR_SET = set(self.PV[:self.NADJ]), set(self.PV[self.NADJ:]); self.MU, self.SEL, self.LO, self.HI = b['mean'], b['selected'], b['hdi_lo'], b['hdi_hi']
    def pat_feats(self, r): return [self.PIX[a] for a in r['adjectives'] if a in self.ADJ_SET] + [self.PIX[p] for p in r['transaction'] if p in self.PAIR_SET]
class Ranker:
    def __init__(self, vec, lexicon, tables):
        self.vec, self.T = vec, tables; self.TAGS = [t for t in lexicon if vec(t) is not None]; self.IX = {t: i for i, t in enumerate(self.TAGS)}
        M = np.vstack([vec(t) for t in self.TAGS]); self.M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
        self.c2w_adj = {c: self.IX[c] for c in tables.EN_CLS if c in self.IX}; self.c2w_pat = {c: self.IX[c] for c in tables.PCLS if c in self.IX}
    def sim(self, terms):
        s = np.zeros(len(self.TAGS), np.float32)
        for t in terms:
            v = self.vec(t)
            if v is None: continue
            sims = self.M @ (v / (np.linalg.norm(v) + 1e-9)); s += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
        return s
    @staticmethod
    def _norm(s): lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    def en_adj(self, r):
        T = self.T; s = np.zeros(len(self.TAGS), np.float32); rc = [T.COEF[:, T.AIX[a]] for a in r['adjectives'] if a in T.AIX]
        if not rc: return s
        agg = np.sum(rc, 0)
        for c, w in self.c2w_adj.items(): s[w] = agg[T.EN_CLS.index(c)]
        return self._norm(s)
    def en_ben(self, r):
        T = self.T; s = np.zeros(len(self.TAGS), np.float32); f = T.pat_feats(r)
        if not f: return s
        agg = (T.MU * T.SEL)[f].sum(0)
        for c, w in self.c2w_pat.items(): s[w] = agg[T.PCLS.index(c)]
        return self._norm(s)
    def scores(self, r, lam=LAM, en='adj', noun_key='context_nouns', naive=False):
        terms = [t for t in r[noun_key] if t.lower() not in ABSTRACT] + (r['adjectives'] if naive else [])
        s = self._norm(self.sim(terms))
        if lam > 0 and en: s = (1 - lam) * s + lam * (self.en_adj(r) if en == 'adj' else self.en_ben(r))
        return s
    def rank(self, r, **kw):
        s = self.scores(r, **kw); g = r['gold_tag']; return int((s >= s[self.IX[g]]).sum()) if g in self.IX else len(self.TAGS) + 1
    def ranks(self, rows, **kw): return np.array([self.rank(r, **kw) for r in rows], float)
    def top(self, r, n=5, **kw):
        s = self.scores(r, **kw); return [(self.TAGS[j], float(s[j])) for j in np.argsort(-s)[:n]]
    def explain(self, r, topn=3, per_tag=3):
        """for each of the top tags: the credible contributing patterns (posterior mean, 94% interval)"""
        T = self.T; s = self.scores(r, en='ben'); f = T.pat_feats(r); out = []
        for j in np.argsort(-s)[:topn]:
            t = self.TAGS[j]; contrib = []
            if t in T.PCLS:
                k = T.PCLS.index(t); contrib = sorted([(T.PV[p], float(T.MU[p, k]), float(T.LO[p, k]), float(T.HI[p, k])) for p in f if T.SEL[p, k] and T.MU[p, k] > 0], key=lambda x: -x[1])[:per_tag]
            out.append((t, float(s[j]), contrib))
        return out
def metrics(ranks, ks=KS):
    r = np.asarray(ranks, float); o = {f'Hits@{k}': float((r <= k).mean()) for k in ks}; o['nDCG@20'] = float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean()); o['MedR'] = float(np.median(r)); return o
def extract_rows(texts, interpreter, batch_size=64):
    """run the extraction stage on raw sentences -> rows shaped like the evaluation files (context_nouns = extractor nouns)"""
    rows = []
    for it in interpreter.pipe(texts, batch_size=batch_size):
        nouns = sorted(set([e['head_lemma'] for e in it.entities] + list(it.context_nouns)))
        rows.append({'sentence': it.sentence, 'context_nouns': nouns, 'ours_nouns': nouns, 'adjectives': sorted({a for e in it.entities for a in e['adjectives']}), 'tag_hints': it.tag_hints, 'transaction': it.transaction})
    return rows
def load_default(ROOT, backbone='w2v'):
    lex = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
    T = Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz')
    cache = {'w2v': 'w2v_googlenews_subset_v2.pkl', 'numberbatch': 'numberbatch_subset.pkl'}[backbone]
    return Ranker(ladder(pickle.load(open(f'{ROOT}/models_cache/{cache}', 'rb'))), lex, T)
'''
open(f'{ROOT}/pipeline_ranker.py', 'w').write(RANKER_SRC.lstrip()); print('wrote ROOT/pipeline_ranker.py', len(RANKER_SRC), 'chars')
import sys, importlib; sys.path.insert(0, ROOT); import pipeline_ranker as pr; importlib.reload(pr)
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); chk = pr.metrics(pr.load_default(ROOT).ranks(ev106)); print('module check on the base-paper set (λ=.35):', {k: round(v, 3) for k, v in chk.items() if k in ('Hits@1', 'Hits@3', 'nDCG@20')}, '<- must be .311 / .528 / .531')

In [ ]:
# N2.1 — translation loss: every input variant through the SAME fully automatic pipeline (transformer extractor → nouns/adjectives → ranker)
import subprocess, sys, importlib, importlib.util, json, numpy as np
if importlib.util.find_spec('en_core_web_trf') is None: subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_trf', '-q'], check=True)
import spacy; spacy.prefer_gpu(); sys.path.insert(0, ROOT); import adjective_extractor as ax, pipeline_ranker as pr
PI = ax.PlaceInterpreter(model='en_core_web_trf'); N1 = json.load(open(f'{ROOT}/results_n/n1_translations.json')); T = N1['sheet']
pk = {r['row']: r for r in json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))}; rows = T['row']; gold = [pk[r]['gold_tag'] for r in rows]
RK = {'W2V': pr.load_default(ROOT, 'w2v'), 'Numberbatch': pr.load_default(ROOT, 'numberbatch')}
VARIANTS = {'English original (collection nouns)': None, 'English original (extractor nouns)': [pk[r]['sentence'] for r in rows],
            'Urdu → NLLB-200 → English': T['en_from_urdu_nllb'], 'Urdu → LLM → English': T['en_from_urdu_llm']}
ru = [i for i, s in enumerate(T['en_from_roman_llm']) if s]; VARIANTS['Roman-Urdu → LLM → English'] = [T['en_from_roman_llm'][i] for i in ru]
RES = {'n_rows': len(rows), 'n_roman': len(ru), 'variants': {}}
def with_gold(extracted, idx):
    for r, i in zip(extracted, idx): r['gold_tag'] = gold[i]; r['row'] = rows[i]
    return extracted
for name, texts in VARIANTS.items():
    idx = ru if name.startswith('Roman') else list(range(len(rows)))
    if texts is None: ev = [{**pk[rows[i]], 'gold_tag': gold[i]} for i in idx]; nk = 'context_nouns'
    else: ev = with_gold(pr.extract_rows(texts, PI), idx); nk = 'ours_nouns'
    RES['variants'][name] = {}
    for bb, rk in RK.items():
        base = rk.ranks(ev, lam=0, noun_key=nk); full = rk.ranks(ev, noun_key=nk)
        RES['variants'][name][bb] = {'nouns_only': pr.metrics(base), 'with_table': pr.metrics(full)}
    m = RES['variants'][name]['W2V']; print(f'{name:<40} n={len(ev):>3}  nouns-only H@3 {m["nouns_only"]["Hits@3"]:.3f}  + table H@3 {m["with_table"]["Hits@3"]:.3f}  nDCG {m["with_table"]["nDCG@20"]:.3f}   | Numberbatch + table H@3 {RES["variants"][name]["Numberbatch"]["with_table"]["Hits@3"]:.3f}')
# same-row comparison for the Roman subset (English originals restricted to the rows that have Roman-Urdu)
if ru:
    ev_en = with_gold(pr.extract_rows([pk[rows[i]]['sentence'] for i in ru], PI), ru)
    RES['english_on_roman_rows'] = {bb: pr.metrics(rk.ranks(ev_en, noun_key='ours_nouns')) for bb, rk in RK.items()}; print(f'{"English original on the Roman-Urdu rows":<40} n={len(ru):>3}  + table H@3 {RES["english_on_roman_rows"]["W2V"]["Hits@3"]:.3f}')
json.dump(RES, open(f'{ROOT}/results_n/n2_translation_loss.json', 'w'), indent=1); print('wrote results_n/n2_translation_loss.json')

In [ ]:
# N3.1 — audit the held bucket, then run the genuinely Urdu sentences through the front-end → results_n/n3_held_urdu.json
import re, pandas as pd
held = pd.read_parquet(f'{ROOT}/clean/held_urdu_bucket.parquet'); tcol = 'text'
held['is_arab'] = held[tcol].astype(str).str.contains(r'[؀-ۿ]')
RU = {'hai','hain','nahi','nahin','kya','yeh','yahan','wahan','bohat','bohot','bahut','acha','accha','khana','jagah','bara','bari','chota','choti','sab','aur','mein','log','raat','zabardast','mazedar','shandar','purana','purani','dekho','milta','hota','karta','shehar','gali','bazar','masjid','qila','bagh','dhaba','nihari','biryani'}
held['ru_hits'] = held[tcol].apply(lambda s: len(set(re.findall(r"[a-z]+", str(s).lower())) & RU))
urdu = held[held.is_arab]; latin = held[~held.is_arab]
AUDIT = {'bucket_rows': len(held), 'urdu_script': int(len(urdu)), 'latin_script': int(len(latin)), 'latin_with_2plus_roman_urdu_markers': int((latin.ru_hits >= 2).sum()),
         'finding': ('The bucket held out by the language router during corpus cleaning is not Urdu/Roman-Urdu as originally labelled. '
                     f'{len(urdu)} rows are genuine Urdu script (Pakistani Wikivoyage edits); the remaining {len(latin)} are English sentences from non-English-speaking destinations '
                     '(Turkish, Thai, Vietnamese, Indian pages) that the router could not classify because of foreign proper nouns. It is a "held non-English / unclassified" bucket.'),
         'latin_sample': [str(t)[:120] for t in latin[tcol].head(8)]}
# front-end on the genuinely Urdu rows — NLLB output for these rows is already in results_n/n1_translations.json
H = N1['held']; idx = [i for i, s in enumerate(H['script']) if s == 'urdu']
en_u = [H['en_nllb'][i] or H['en_llm'][i] for i in idx]; src_u = [H['text'][i] for i in idx]
ev = pr.extract_rows(en_u, PI) if en_u else []; rk = RK['W2V']; out = []
for i, r in enumerate(ev):
    q = {**r, 'context_nouns': r['ours_nouns']}; ex = rk.explain(q); top = rk.top(q, n=3)
    out.append({'urdu': src_u[i], 'english': en_u[i], 'nouns': r['ours_nouns'], 'adjectives': r['adjectives'],
                'top3': [t for t, _ in top], 'explanation': [(t, [(p, round(m, 2), round(lo, 2), round(hi, 2)) for p, m, lo, hi in c]) for t, _, c in ex]})
json.dump({'audit': AUDIT, 'urdu_rows': out, 'translator': 'NLLB-200-distilled-600M (urd_Arab→eng_Latn)'}, open(f'{ROOT}/results_n/n3_held_urdu.json', 'w'), ensure_ascii=False, indent=1)
print(AUDIT['finding'])
print(f'\nran the front-end on the {len(out)} Urdu-script rows:')
for o in out: print(f'\n  {o["urdu"][:70]}\n    → {o["english"][:95]}\n    nouns {o["nouns"][:6]} adj {o["adjectives"][:4]} → top-3 {o["top3"]}')

In [ ]:
# N3.2 — what is actually in the "held Urdu" bucket? (the 230 Latin-script rows look like other languages, not Roman Urdu)
import re, pandas as pd
held = pd.read_parquet(f'{ROOT}/clean/held_urdu_bucket.parquet'); tcol = 'text'
print('lang column values:', held['lang'].value_counts().to_dict() if 'lang' in held.columns else 'none')
RU = {'hai','hain','nahi','nahin','kya','yeh','yahan','wahan','bohat','bohot','bahut','acha','accha','khana','jagah','bara','bari','chota','choti','sab','aur','ka','ki','ke','se','mein','par','ko','ye','wo','log','din','raat','sahi','zabardast','mazay','mazedar','shandar','purana','purani','naya','nai','dekho','jao','milta','milti','hota','hoti','karta','karti','pani','shehar','gali','bazar','masjid','qila','bagh','dhaba','nihari','biryani','chai'}
def ru_score(s):
    w = set(re.findall(r"[a-z']+", str(s).lower())); return len(w & RU)
held['ru_hits'] = held[tcol].apply(ru_score); held['is_arab'] = held[tcol].astype(str).str.contains(r'[؀-ۿ]')
print(f"\nUrdu script rows: {int(held.is_arab.sum())}")
for t in held[held.is_arab][tcol].head(5): print('   ', str(t)[:100])
lat = held[~held.is_arab]
print(f"\nLatin-script rows: {len(lat)} | with >=2 Roman-Urdu marker words: {int((lat.ru_hits >= 2).sum())} | >=1: {int((lat.ru_hits >= 1).sum())}")
print('\nsample Latin-script rows (ru_hits, place_name, text):')
for _, r in lat.sample(8, random_state=0).iterrows(): print(f"   [{r.ru_hits}] {str(r.place_name)[:18]:<18} {str(r[tcol])[:95]}")
print('\ntop place_name in the Latin-script rows:'); print(lat.place_name.value_counts().head(12).to_string())

In [ ]:
# N3.3 — correct the corpus-composition record: the held bucket was mislabelled "Urdu / Roman-Urdu" (see the N3.1 audit)
cc_path = f'{ROOT}/results_i/corpus_composition.json'; cc = json.load(open(cc_path)); A = json.load(open(f'{ROOT}/results_n/n3_held_urdu.json'))['audit']
cc['held_unclassified'] = cc.pop('held_urdu_roman_urdu', A['bucket_rows'])          # what the language router actually held out
cc['held_urdu_script'] = A['urdu_script']; cc['held_note'] = A['finding']
json.dump(cc, open(cc_path, 'w'), indent=2); print({k: cc[k] for k in ('routed_english', 'sentences_after_split', 'held_unclassified', 'held_urdu_script')})